# NetSentinel V2 — Inspector–Sentry harness

Trains the **Inspector** (E-GraphSAGE + Transformer denoising autoencoder), distils a
**Sentry** ~14× smaller, and races three routers at equal escalation budget. Then runs the
**distribution-shift stress test** and, optionally, loads **real Zeek traffic**.

---
### Read this before you turn on a GPU

**You almost certainly don't need one.** The default run is ~2 min/seed on CPU. A GPU helps
only once you scale up (`--hosts 2000`, bigger encoder, real capture). Turn it on when the
cell timings tell you to, not before — Kaggle's 30 GPU-hours/week are worth saving.

### Kaggle settings you do need

| Setting | Value | Why |
|---|---|---|
| Accelerator | **None** (CPU) to start | the harness is CPU-bound and small |
| Internet | **On** | only for the real-data section (needs phone verification) |
| Persistence | Variables & files | keeps `results.json` between sessions |

### Honest scope

The generator is a **stand-in, not a dataset**. Router-vs-Inspector agreement is meaningful
(the Inspector is the reference oracle); **attack-recall numbers are indicative only** until
real capture replaces it. See `DATASETS.md`.

In [ ]:
import sys, platform, os, time
import numpy as np
try:
    import torch
    print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('   ', torch.cuda.get_device_name(0))
except ImportError:
    print('installing torch...'); os.system('pip -q install torch')
print('python', platform.python_version(), '| cpus', os.cpu_count())
os.makedirs('/kaggle/working/netsentinel_v2', exist_ok=True)
os.chdir('/kaggle/working')
print('cwd', os.getcwd())

## 1 · Write the package

Self-contained on purpose — no upload step, no GitHub dependency. Re-run these cells after
editing anything.

In [ ]:
%%writefile netsentinel_v2/categories.py
"""Service Category Resolver — taxonomy and the edge-feature schema.

The GNN never sees a domain. It sees a *category* and the shape of the
communication with it. That is what makes the model learn the behavioural
concept of a kill chain instead of memorising infrastructure.
"""

from __future__ import annotations

# --- Category taxonomy -----------------------------------------------------
# Order is fixed and load-bearing: it indexes the per-window edge tensor.
CATEGORIES = [
    "Recon_API",        # ip-api.com, ipify.org        -> victim fingerprinting
    "Code_Repo_Paste",  # raw.githubusercontent, pastebin -> stager pull
    "Messaging_API",    # api.telegram.org, discord     -> C2 polling
    "Cloud_Storage",    # drive.google.com, dropbox     -> exfil (T1567)
    "Browse",           # general web
    "Sync",             # OneDrive/Dropbox desktop agents
    "CI_CD",            # build runners, artifact stores
    "Internal",         # east-west (the OT/diode generalisation lives here)
    "Unknown_External", # resolver could not categorise (ECH/DoH fallback)
]
N_CATEGORIES = len(CATEGORIES)
CAT_INDEX = {c: i for i, c in enumerate(CATEGORIES)}

# --- Edge feature schema ---------------------------------------------------
# One row per (host, category) per window. Features are deliberately the ones
# CRITIQUE B4 says are evadable, PLUS the distribution-family tests that turn
# the evasion itself into a signature (the "Jitter-Trap" inversion).
EDGE_FEATURES = [
    "log_n_flows",             # volume
    "log_bytes_up",
    "log_bytes_down",
    "egress_asymmetry",        # (up-down)/(up+down)  -> exfil signal
    "iat_cv",                  # classic beacon CoV (weak on its own)
    "ks_uniform",              # KS stat vs Uniform  -> LOW means jittered beacon
    "ks_exponential",          # KS stat vs Exponential -> LOW means human/Poisson
    "fft_prominence",          # dominant-frequency prominence -> automation
    "distinct_endpoint_ratio", # ->1.0 means URL/domain randomiser
    "log_duration_mean",
]
N_EDGE_FEATURES = len(EDGE_FEATURES)

# Which categories a role is *expected* to reach (the pre-path model).
# Off-pre-path traffic is the primary signal, but only in conjunction with
# the automation and egress gates (CRITIQUE B3).
ROLE_PREPATH = {
    "developer":  {"Code_Repo_Paste", "CI_CD", "Messaging_API", "Cloud_Storage",
                   "Browse", "Sync", "Internal"},
    "office":     {"Browse", "Sync", "Cloud_Storage", "Messaging_API", "Internal"},
    "kiosk":      {"Browse", "Internal"},
    "ot_hmi":     {"Internal"},
    "server":     {"CI_CD", "Internal", "Sync"},
}
ROLES = list(ROLE_PREPATH)

In [ ]:
%%writefile netsentinel_v2/synth.py
"""Synthetic host-day generator.

>>> HONESTY NOTE — read before quoting any number produced from this file. <<<
This is a STAND-IN, not a dataset. A detector evaluated on the generator that
made its attacks learns the generator, not the adversary (CRITIQUE A5/B2).
Its only legitimate uses are:
  (1) exercising the harness end-to-end before real data exists, and
  (2) the teacher-vs-student agreement experiment, which does NOT depend on
      the attacks being realistic because the teacher is the reference oracle.
Detection numbers against `is_attack` are indicative ONLY. Swap in real
benign capture + Mythic-emulated chains before reporting anything externally.

What it does try to get right, because these decide whether the experiment is
meaningful at all:
  * benign behaviour is MULTIMODAL (workday / evening / weekend) -- G4
  * DevOps hosts legitimately produce Recon->Repo->Messaging->Cloud -- B3
    with HUMAN timing. These are the hard negatives. Without them the
    experiment is trivially easy and the result is worthless.
  * edge features are COMPUTED from generated inter-arrival times, not
    invented, so the KS/FFT automation features are real statistics.
"""

from __future__ import annotations

import numpy as np
from scipy import stats

from .categories import (
    CAT_INDEX, N_CATEGORIES, N_EDGE_FEATURES, ROLES, ROLE_PREPATH,
)

WINDOWS_PER_DAY = 24  # one-hour windows

# --------------------------------------------------------------------------
# "Worlds" — two organisations with the SAME mechanism but different shape.
# World A is what the encoder trains on. World B is a different customer:
# different role mix, different human rhythms, different data volumes.
# Deploying an encoder trained on A into B, with baselines re-fit on B, is
# exactly the distribution shift a real deployment faces. If the learned
# "normal" is brittle, this is where it shows.
# --------------------------------------------------------------------------
WORLDS = {
    "A": dict(role_probs=[.30, .34, .10, .14, .12],   # dev-heavy software org
              human_sigma=1.10, iat_scale=1.00, volume=1.00,
              devops_p=0.35, offhours=0.25, work_lo=9, work_hi=18),
    "B": dict(role_probs=[.10, .30, .18, .30, .12],   # OT/plant-heavy, few devs
              human_sigma=1.55, iat_scale=2.20, volume=0.45,
              devops_p=0.12, offhours=0.60, work_lo=6, work_hi=22),
}


def world(name_or_dict):
    if name_or_dict is None:
        return dict(WORLDS["A"])
    if isinstance(name_or_dict, str):
        return dict(WORLDS[name_or_dict])
    w = dict(WORLDS["A"]); w.update(name_or_dict); return w


# --------------------------------------------------------------------------
# Edge features are derived from an actual IAT sample, so ks_uniform /
# ks_exponential / fft_prominence mean what they claim to mean.
# --------------------------------------------------------------------------
def _ks(x_sorted: np.ndarray, cdf: np.ndarray) -> float:
    """One-sample KS statistic, vectorised (scipy.kstest is ~100x slower and
    this is called once per host-window-category)."""
    n = x_sorted.size
    i = np.arange(1, n + 1)
    return float(np.maximum(i / n - cdf, cdf - (i - 1) / n).max())


def _iat_stats(iats: np.ndarray) -> tuple[float, float, float, float]:
    """Return (cv, ks_uniform, ks_exponential, fft_prominence)."""
    if iats.size < 4:
        return 0.0, 1.0, 1.0, 0.0
    mu = float(iats.mean())
    if mu <= 0:
        return 0.0, 1.0, 1.0, 0.0
    cv = float(iats.std() / mu)

    xs = np.sort(iats)
    lo, hi = float(xs[0]), float(xs[-1])
    if hi - lo < 1e-9:
        ks_u = 1.0                      # constant: maximally un-uniform
    else:
        ks_u = _ks(xs, np.clip((xs - lo) / (hi - lo), 0, 1))
    ks_e = _ks(xs, 1.0 - np.exp(-xs / mu))

    # Periodicity: prominence of the dominant non-DC frequency of the
    # pulse train implied by the IATs.
    n = min(iats.size, 128)
    x = iats[:n] - iats[:n].mean()
    if np.allclose(x, 0):
        prom = 1.0                      # perfectly periodic
    else:
        mag = np.abs(np.fft.rfft(x))[1:]
        prom = float(mag.max() / (mag.mean() + 1e-9) / n) if mag.size else 0.0
        prom = min(prom, 1.0)
    return cv, ks_u, ks_e, prom


def _edge_row(rng, n_flows, bytes_up, bytes_down, iats, distinct_ratio, dur_mean):
    cv, ks_u, ks_e, prom = _iat_stats(iats)
    tot = bytes_up + bytes_down + 1.0
    return np.array([
        np.log1p(n_flows),
        np.log1p(bytes_up),
        np.log1p(bytes_down),
        (bytes_up - bytes_down) / tot,
        cv,
        ks_u,
        ks_e,
        prom,
        distinct_ratio,
        np.log1p(dur_mean),
    ], dtype=np.float32)


def _human_iats(rng, n, scale, sigma=1.1):
    """Humans are bursty / heavy-tailed, not uniform and not constant."""
    return rng.lognormal(mean=np.log(max(scale, 1e-6)), sigma=sigma, size=n)


def _beacon_iats(rng, n, sleep, jitter):
    """Beacon with jitter -> UNIFORM IATs. The evasion is the signature."""
    return rng.uniform(sleep * (1 - jitter), sleep * (1 + jitter), size=n)


# --------------------------------------------------------------------------
def _benign_window(rng, role, hour, is_weekend, devops_chain_today, W):
    """One (N_CATEGORIES, N_EDGE_FEATURES) window plus a presence mask."""
    edges = np.zeros((N_CATEGORIES, N_EDGE_FEATURES), dtype=np.float32)
    mask = np.zeros(N_CATEGORIES, dtype=np.float32)

    working = (W["work_lo"] <= hour <= W["work_hi"]) and not is_weekend
    # Multimodality: intensity depends on hour AND weekday (G4).
    intensity = (1.0 if working else (W["offhours"] if 7 <= hour <= 22 else 0.05))
    prepath = ROLE_PREPATH[role]
    V = W["volume"]

    for cat in prepath:
        # not every allowed category is touched every window
        p = 0.75 * intensity if cat in ("Browse", "Internal") else 0.35 * intensity
        if rng.random() > p:
            continue
        ci = CAT_INDEX[cat]
        n = max(2, int(rng.gamma(3.0, 6.0 * intensity)))
        iats = _human_iats(rng, n, 3600.0 / max(n, 1) * W["iat_scale"], W["human_sigma"])

        if cat == "Sync":
            up, down = rng.gamma(2, 4e5*V), rng.gamma(2, 8e5*V)
            # sync agents are semi-periodic -> genuinely confusable, keep it
            iats = rng.normal(300, 60, size=n).clip(30, None)
        elif cat == "Cloud_Storage":
            up, down = rng.gamma(2, 3e5*V), rng.gamma(2, 6e5*V)
        elif cat == "Code_Repo_Paste":
            up, down = rng.gamma(2, 2e4*V), rng.gamma(2, 9e5*V)
        elif cat == "CI_CD":
            up, down = rng.gamma(2, 1e5*V), rng.gamma(2, 5e5*V)
            iats = rng.normal(600, 240, size=n).clip(30, None)  # scheduled jobs
        elif cat == "Messaging_API":
            up, down = rng.gamma(2, 8e3*V), rng.gamma(2, 3e4*V)
        else:
            up, down = rng.gamma(2, 3e4*V), rng.gamma(2, 4e5*V)

        edges[ci] = _edge_row(rng, n, up, down, iats,
                              distinct_ratio=rng.uniform(0.05, 0.45),
                              dur_mean=rng.gamma(2, 4))
        mask[ci] = 1.0

    # ---- HARD NEGATIVE (B3): a real DevOps chain, human-timed --------------
    # Same category sequence as the attack. Different shape. If the model
    # cannot separate these, the category-transition hypothesis is dead and
    # the experiment must say so.
    if devops_chain_today and working and role == "developer":
        stage = {10: "Recon_API", 11: "Code_Repo_Paste",
                 12: "Messaging_API", 14: "Cloud_Storage"}.get(hour)
        if stage:
            ci = CAT_INDEX[stage]
            n = max(3, int(rng.gamma(3.0, 5.0)))
            iats = _human_iats(rng, n, 200.0 * W["iat_scale"], W["human_sigma"])
            if stage == "Cloud_Storage":
                up, down = rng.gamma(2, 8e5), rng.gamma(2, 3e5)  # real upload!
            elif stage == "Recon_API":
                up, down = rng.gamma(2, 2e3), rng.gamma(2, 4e3)
            elif stage == "Code_Repo_Paste":
                up, down = rng.gamma(2, 2e4), rng.gamma(2, 1e6)
            else:
                up, down = rng.gamma(2, 1e4), rng.gamma(2, 4e4)
            edges[ci] = _edge_row(rng, n, up, down, iats,
                                  distinct_ratio=rng.uniform(0.1, 0.5),
                                  dur_mean=rng.gamma(2, 5))
            mask[ci] = 1.0
    return edges, mask


def _inject_lsa(rng, edges, mask, hour, stage_hours, stealth):
    """Overlay one LSA stage. `stealth` in [0,1]: 1.0 = fully shaped adversary.

    A high-stealth attacker mimics human IAT distributions and dribbles the
    exfil, which is exactly the adversary CRITIQUE B4 says defeats the timing
    channel. Keeping it in the generator is the only way the experiment can
    report an honest degradation curve.
    """
    stage = stage_hours.get(hour)
    if stage is None:
        return False
    ci = CAT_INDEX[stage]

    if stage == "Recon_API":
        n = rng.integers(1, 4)
        iats = _human_iats(rng, n, 60) if stealth > .5 else _beacon_iats(rng, n, 30, .1)
        up, down = rng.gamma(2, 5e2), rng.gamma(2, 1.5e3)
        dr = rng.uniform(0.8, 1.0)
    elif stage == "Code_Repo_Paste":
        n = rng.integers(1, 3)
        iats = _human_iats(rng, n, 120)
        up, down = rng.gamma(2, 1e3), rng.gamma(2, 2e5)
        dr = rng.uniform(0.6, 1.0)
    elif stage == "Messaging_API":
        # C2 polling: the core beacon. Jitter grows with stealth.
        n = max(6, int(rng.gamma(6.0, 6.0)))
        jitter = 0.05 + 0.55 * stealth
        iats = (_human_iats(rng, n, 45) if stealth > 0.85
                else _beacon_iats(rng, n, 45, jitter))
        up, down = rng.gamma(2, 6e2), rng.gamma(2, 9e2)   # tiny both ways
        dr = rng.uniform(0.7, 1.0)
    else:  # Cloud_Storage exfil
        n = rng.integers(1, 5)
        iats = _human_iats(rng, n, 90)
        # stealth dribbles the upload -> defeats egress asymmetry gate
        vol = 6e6 * (1.0 - 0.92 * stealth)
        up, down = rng.gamma(2, vol), rng.gamma(2, 2e4)
        dr = rng.uniform(0.5, 0.9)

    row = _edge_row(rng, n, up, down, np.asarray(iats, dtype=float), dr,
                    dur_mean=rng.gamma(2, 2))
    if mask[ci] > 0:                      # blend into existing benign traffic
        edges[ci] = 0.5 * edges[ci] + 0.5 * row
    else:
        edges[ci] = row
    mask[ci] = 1.0
    return True


def generate(n_hosts=300, n_days=24, attack_host_frac=0.08, seed=0,
             stealth_range=(0.0, 1.0), world_cfg="A"):
    """Return dict of arrays.

    edges : (H, D, W, C, F)   host-day-window-category-feature
    mask  : (H, D, W, C)
    is_attack_day : (H, D) bool
    stealth       : (H,) float, 0 for benign hosts
    roles, host_ids
    """
    rng = np.random.default_rng(seed)
    W = world(world_cfg)
    roles = rng.choice(ROLES, size=n_hosts, p=W["role_probs"])  # dev/office/kiosk/ot/server
    n_attack = max(1, int(n_hosts * attack_host_frac))
    attack_hosts = rng.choice(n_hosts, size=n_attack, replace=False)
    stealth = np.zeros(n_hosts, dtype=np.float32)
    stealth[attack_hosts] = rng.uniform(*stealth_range, size=n_attack)
    # attacker activates partway through, after commissioning -> dormant-then-active
    activate_day = rng.integers(n_days // 2, n_days, size=n_hosts)

    E = np.zeros((n_hosts, n_days, WINDOWS_PER_DAY, N_CATEGORIES, N_EDGE_FEATURES),
                 dtype=np.float32)
    M = np.zeros((n_hosts, n_days, WINDOWS_PER_DAY, N_CATEGORIES), dtype=np.float32)
    A = np.zeros((n_hosts, n_days), dtype=bool)

    stage_hours = {9: "Recon_API", 10: "Code_Repo_Paste",
                   **{h: "Messaging_API" for h in range(11, 20)},
                   20: "Cloud_Storage"}

    for h in range(n_hosts):
        role = roles[h]
        is_attacker = stealth[h] > 0 or h in attack_hosts
        for d in range(n_days):
            weekend = (d % 7) in (5, 6)
            devops_today = (role == "developer") and (rng.random() < W["devops_p"])
            attacking = is_attacker and d >= activate_day[h]
            hit = False
            for w in range(WINDOWS_PER_DAY):
                e, m = _benign_window(rng, role, w, weekend, devops_today, W)
                if attacking:
                    hit |= _inject_lsa(rng, e, m, w, stage_hours, float(stealth[h]))
                E[h, d, w] = e
                M[h, d, w] = m
            A[h, d] = attacking and hit

    return dict(edges=E, mask=M, is_attack_day=A, stealth=stealth,
                roles=roles, activate_day=activate_day,
                host_ids=np.arange(n_hosts), world=W)

In [ ]:
%%writefile netsentinel_v2/models.py
"""Inspector (teacher) and Sentry (student) encoders.

Inspector : edge-aware GNN over (host -> service-category) star graphs, with a
            sampled hop-2 cohort context, then a Transformer over the day's
            window sequence, trained as a denoising graph autoencoder.
            This is the GraphIDS-shaped architecture. It is PRIOR ART
            (arXiv:2509.16625) -- it is our Inspector, not our contribution.

Sentry    : same interface, ~10x smaller, NO hop-2, GRU instead of Transformer.
            Trained by distillation from the Inspector's ENCODER, never from
            its decision. That split is the whole point (see V2_HARDENING B1).
"""

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

from .categories import N_CATEGORIES, N_EDGE_FEATURES


def _mlp(i, h, o):
    return nn.Sequential(nn.Linear(i, h), nn.GELU(), nn.Linear(h, o))


class EdgeAggregator(nn.Module):
    """E-GraphSAGE-style hop-1 aggregation over a host's category edges.

    message(c) = MLP([edge_features(c) , category_embedding(c) , cohort(c)])
    window     = [ mean_c message , max_c message , presence_mask ]

    The category embedding is what stops the model memorising infrastructure;
    the cohort term is the sampled hop-2 neighbourhood (other hosts touching
    the same category in the same window).
    """

    def __init__(self, dim, use_cohort=True):
        super().__init__()
        self.use_cohort = use_cohort
        self.cat_emb = nn.Embedding(N_CATEGORIES, 16)
        in_dim = N_EDGE_FEATURES + 16 + (N_EDGE_FEATURES if use_cohort else 0)
        self.msg = _mlp(in_dim, dim, dim)
        self.out = _mlp(2 * dim + N_CATEGORIES, dim, dim)

    def forward(self, edges, mask, cohort=None):
        # edges (B,W,C,F)  mask (B,W,C)  cohort (B,W,C,F)
        B, W, C, Fdim = edges.shape
        ce = self.cat_emb.weight.view(1, 1, C, -1).expand(B, W, C, -1)
        parts = [edges, ce]
        if self.use_cohort:
            parts.append(cohort if cohort is not None else torch.zeros_like(edges))
        m = self.msg(torch.cat(parts, dim=-1))              # (B,W,C,d)

        mk = mask.unsqueeze(-1)                             # (B,W,C,1)
        cnt = mk.sum(dim=2)                                 # (B,W,1) TRUE count
        denom = cnt.clamp(min=1.0)
        mean = (m * mk).sum(dim=2) / denom
        # An empty window has no edges to max over. Guarding on `denom` here is
        # wrong -- it is clamped to >=1 and so is never 0, which let the -1e9
        # sentinel escape into the encoder on every empty window (~half of them)
        # and blew the first-epoch loss up to ~1e12. Guard on the true count.
        mx = (m.masked_fill(mk == 0, -1e9)).max(dim=2).values
        mx = torch.where(cnt > 0, mx, torch.zeros_like(mx))
        return self.out(torch.cat([mean, mx, mask], dim=-1))  # (B,W,d)


class Inspector(nn.Module):
    """Expensive teacher. Runs during commissioning and on re-escalation only."""

    def __init__(self, dim=96, heads=4, layers=2):
        super().__init__()
        self.dim = dim
        self.agg = EdgeAggregator(dim, use_cohort=True)
        self.pos = nn.Parameter(torch.randn(1, 24, dim) * 0.02)
        enc = nn.TransformerEncoderLayer(dim, heads, dim * 2, dropout=0.1,
                                         batch_first=True, norm_first=True)
        self.seq = nn.TransformerEncoder(enc, layers)
        self.dec = _mlp(dim + 16, dim, N_EDGE_FEATURES)
        self.cat_emb = nn.Embedding(N_CATEGORIES, 16)

    def encode(self, edges, mask, cohort):
        h = self.agg(edges, mask, cohort) + self.pos[:, : edges.shape[1]]
        return self.seq(h)                                   # (B,W,d)

    def reconstruct(self, z):
        B, W, d = z.shape
        ce = self.cat_emb.weight.view(1, 1, N_CATEGORIES, 16).expand(B, W, -1, -1)
        zz = z.unsqueeze(2).expand(B, W, N_CATEGORIES, d)
        return self.dec(torch.cat([zz, ce], dim=-1))         # (B,W,C,F)

    def forward(self, edges, mask, cohort):
        z = self.encode(edges, mask, cohort)
        return z, self.reconstruct(z)


class Sentry(nn.Module):
    """Cheap always-on student. No cohort (no hop-2), GRU not Transformer."""

    def __init__(self, dim=32, teacher_dim=96):
        super().__init__()
        self.dim = dim
        self.agg = EdgeAggregator(dim, use_cohort=False)
        self.seq = nn.GRU(dim, dim, batch_first=True, bidirectional=False)
        self.project = nn.Linear(dim, teacher_dim)   # distillation head only

    def encode(self, edges, mask):
        h = self.agg(edges, mask)
        out, _ = self.seq(h)
        return out                                    # (B,W,dim)

    def forward(self, edges, mask):
        z = self.encode(edges, mask)
        return z, self.project(z)


class DeferralHead(nn.Module):
    """G5 fix: predict *"would the Inspector flag this window?"* directly.

    "Sentry escalates when it is unsure" is confidence-based deferral, which is
    provably suboptimal under distribution shift, specialist downstream models
    and label noise (Jitkrittum et al., NeurIPS 2023) -- all three describe an
    IDS. So we do not threshold the Sentry's own score. We train a head on the
    Sentry embedding + the per-host baseline distance to predict the teacher's
    decision. The label is free: during commissioning we ran both models.
    """

    def __init__(self, sentry_dim=32, n_extra=3):
        super().__init__()
        self.net = _mlp(sentry_dim + n_extra, 64, 1)

    def forward(self, z_sentry, extra):
        return self.net(torch.cat([z_sentry, extra], dim=-1)).squeeze(-1)


def recon_error(pred, edges, mask):
    """Per-window mean squared reconstruction error over PRESENT edges."""
    se = ((pred - edges) ** 2).mean(dim=-1)               # (B,W,C)
    mk = mask
    return (se * mk).sum(dim=2) / mk.sum(dim=2).clamp(min=1.0)   # (B,W)

In [ ]:
%%writefile netsentinel_v2/baseline.py
"""Per-host behavioural baseline in Sentry-embedding space.

This is the part that is deliberately NOT distilled. Distillation degrades
local-outlier detection (20% transfer) and neighbourhood methods (76%
retention) while preserving global structure (78-88%) -- so we distil the
*representation* and keep the *local decision* exact.

Three corrections from V2_HARDENING Part G are implemented here:

  G3  Ledoit-Wolf shrinkage. A 14-day commissioning window gives ~O(d)
      samples, not the >> d^2 a raw sample covariance needs. Without
      shrinkage Sigma is singular and the distance is meaningless.

  G4  Multimodality. One Gaussian cannot cover workday / evening / weekend.
      We fit k components and score against the nearest one.

  Attacker-in-the-baseline. Windows with the highest teacher error are
      trimmed before fitting, so a slow attacker present during commissioning
      contributes less to "normal". This cuts both ways -- it also removes
      legitimate rare behaviour, which is then flagged forever after. That
      trade is real and is reported, not hidden.
"""

from __future__ import annotations

import numpy as np
from sklearn.covariance import LedoitWolf
from sklearn.cluster import KMeans


class HostBaseline:
    def __init__(self, n_components=3, trim_frac=0.05, min_per_component=12):
        self.k = n_components
        self.trim_frac = trim_frac
        self.min_per_component = min_per_component
        self.means_: list[np.ndarray] = []
        self.precisions_: list[np.ndarray] = []
        self.fitted_ = False

    def fit(self, Z: np.ndarray, teacher_err: np.ndarray | None = None):
        """Z (n, d) commissioning embeddings; teacher_err (n,) for trimming."""
        Z = np.asarray(Z, dtype=np.float64)
        if teacher_err is not None and self.trim_frac > 0 and len(Z) > 20:
            keep = np.argsort(teacher_err)[: max(10, int(len(Z) * (1 - self.trim_frac)))]
            Z = Z[keep]

        n, d = Z.shape
        k = max(1, min(self.k, n // max(self.min_per_component, 1)))
        if k == 1:
            labels = np.zeros(n, dtype=int)
        else:
            labels = KMeans(n_clusters=k, n_init=4, random_state=0).fit_predict(Z)

        self.means_, self.precisions_ = [], []
        for c in range(k):
            Zc = Z[labels == c]
            if len(Zc) < 4:
                Zc = Z                       # degenerate cluster -> use all
            lw = LedoitWolf(assume_centered=False).fit(Zc)
            self.means_.append(lw.location_)
            self.precisions_.append(lw.precision_)
        self.fitted_ = True
        return self

    def score(self, Z: np.ndarray) -> np.ndarray:
        """Min Mahalanobis distance over components. Higher = more deviant."""
        if not self.fitted_:
            return np.zeros(len(Z))
        Z = np.asarray(Z, dtype=np.float64)
        best = None
        for mu, P in zip(self.means_, self.precisions_):
            delta = Z - mu
            d2 = np.einsum("ij,jk,ik->i", delta, P, delta)
            d2 = np.sqrt(np.clip(d2, 0, None))
            best = d2 if best is None else np.minimum(best, d2)
        return best


def fit_all(Z_by_host: dict, err_by_host: dict | None = None, **kw):
    out = {}
    for h, Z in Z_by_host.items():
        e = None if err_by_host is None else err_by_host.get(h)
        out[h] = HostBaseline(**kw).fit(Z, e)
    return out

In [ ]:
%%writefile netsentinel_v2/diagnostics.py
"""G1 go/no-go diagnostic: does the student's embedding space preserve the
teacher's LOCAL geometry?

The whole B1 fix rests on an assumption I flagged as unproven in
V2_HARDENING G1: moving the local decision out of the network only helps if
the distilled ENCODER still preserves local geometry. Compression is exactly
what destroys local structure, so this must be measured, not assumed.

The right test is NOT "do teacher and student scores correlate". It is
whether a point's NEIGHBOURS are the same in both spaces. If host h's nearest
neighbours differ between encoders, then a Mahalanobis test computed in the
student space is being computed in a geometry the teacher would not
recognise, and B1 is unfixed no matter how well the scores correlate.

Reported:
  knn_overlap@k  mean |N_t(x) ∩ N_s(x)| / k        (1.0 = geometry preserved)
  trustworthiness (sklearn)                        standard manifold measure
  spearman(score_t, score_s)                       the WEAKER test, for contrast
"""

from __future__ import annotations

import numpy as np
from scipy.stats import spearmanr
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import trustworthiness


def knn_overlap(Zt: np.ndarray, Zs: np.ndarray, k: int = 20,
                sample: int = 3000, seed: int = 0) -> float:
    n = len(Zt)
    rng = np.random.default_rng(seed)
    idx = rng.choice(n, size=min(sample, n), replace=False)

    nt = NearestNeighbors(n_neighbors=k + 1).fit(Zt)
    ns = NearestNeighbors(n_neighbors=k + 1).fit(Zs)
    _, it = nt.kneighbors(Zt[idx])
    _, is_ = ns.kneighbors(Zs[idx])

    ov = [len(set(a[1:]) & set(b[1:])) / k for a, b in zip(it, is_)]
    return float(np.mean(ov))


def geometry_report(Zt, Zs, score_t, score_s, k=20, sample=3000):
    Zt, Zs = np.asarray(Zt), np.asarray(Zs)
    n = min(len(Zt), 1500)
    rng = np.random.default_rng(0)
    sub = rng.choice(len(Zt), size=n, replace=False)
    return {
        "knn_overlap@%d" % k: knn_overlap(Zt, Zs, k=k, sample=sample),
        "trustworthiness": float(trustworthiness(Zt[sub], Zs[sub], n_neighbors=min(k, n - 2))),
        "spearman_score": float(spearmanr(score_t, score_s).statistic),
        "n": int(len(Zt)),
    }


# Thresholds are PROVISIONAL. They were asserted before the first run, not
# calibrated. The v1 experiment measured overlap ~0.32 while the Mahalanobis
# router was nonetheless the best of the three -- so partial geometry
# preservation is evidently enough for the baseline test to beat a distilled
# detector. Treat these bands as a monitor for regression, not as physics, and
# recalibrate them against router AUC as more runs accumulate.
GO_BAND, MARGINAL_BAND = 0.50, 0.25


def verdict(report, k=20):
    """Plain-language go/no-go so nobody has to interpret the number."""
    ov = report["knn_overlap@%d" % k]
    if ov >= GO_BAND:
        return ("GO", "Student preserves the teacher's local geometry "
                      "(overlap %.2f). Mahalanobis in student space is well founded." % ov)
    if ov >= MARGINAL_BAND:
        return ("MARGINAL", "Partial geometry preservation (overlap %.2f). The Mahalanobis "
                            "router depends on this geometry and came LAST in the measured "
                            "comparison, with ~7x the seed variance of the learned heads -- "
                            "consistent with only partial preservation. Report the overlap "
                            "alongside any recall claim, and re-check it whenever student "
                            "capacity or the distillation loss changes." % ov)
    return ("NO-GO", "Student does NOT preserve local geometry (overlap %.2f). "
                     "The B1 fix is not established: the local decision has been "
                     "moved into an encoder that destroyed the locality. "
                     "Increase student capacity or distil with a neighbourhood-"
                     "preserving loss before proceeding." % ov)

In [ ]:
%%writefile netsentinel_v2/train.py
"""End-to-end experiment: does the Sentry route well enough to be worth it?

Evaluation protocol follows arXiv:2608.01454 (provenance-IDS benchmark study):
  * strictly monotonic train -> validate -> test split IN TIME (no random split)
  * every threshold calibrated on VALIDATION only, never on test
  * multi-seed, mean +/- std reported

Three routers are compared at equal escalation budget:
  A  distilled DETECTOR  -- student regresses the teacher's anomaly score,
                            escalate on its own score. This is the ORIGINAL
                            V2 design and the one the distillation literature
                            says degrades on local outliers.
  B  encoder + Mahalanobis -- V2_HARDENING B1: distil only the encoder, keep
                            the local decision exact against a per-host
                            baseline.
  C  deferral head       -- V2_HARDENING G5: predict "would the Inspector
                            flag this window", not "am I unsure".
"""

from __future__ import annotations

import json
import time
import numpy as np
import torch
import torch.nn as nn

from .categories import N_CATEGORIES, N_EDGE_FEATURES
from .models import Inspector, Sentry, DeferralHead, recon_error, _mlp
from .baseline import HostBaseline
from .diagnostics import geometry_report, verdict

DEV = torch.device("cpu")


# ---------------------------------------------------------------- utilities
def compute_cohort(E, M):
    """Hop-2 context: mean edge features of OTHER hosts on the same
    (day, window, category). This is the sampled-neighbourhood term that
    E-GraphSAGE contributes over a per-host flat model."""
    s = (E * M[..., None]).sum(axis=0)          # (D,W,C,F)
    n = M.sum(axis=0)[..., None]                # (D,W,C,1)
    own = E * M[..., None]
    cnt = np.maximum(n - M[..., None], 1.0)
    return ((s[None] - own) / cnt).astype(np.float32)


def standardise(E, M, ref_mean=None, ref_std=None):
    flat = E[M > 0]
    if ref_mean is None:
        ref_mean, ref_std = flat.mean(0), flat.std(0) + 1e-6
    out = (E - ref_mean) / ref_std
    return (out * M[..., None]).astype(np.float32), ref_mean, ref_std


def batches(n, bs, shuffle=True, seed=0):
    idx = np.arange(n)
    if shuffle:
        np.random.default_rng(seed).shuffle(idx)
    for i in range(0, n, bs):
        yield idx[i:i + bs]


def t(x):
    return torch.as_tensor(x, dtype=torch.float32, device=DEV)


# ---------------------------------------------------------------- training
def train_inspector(E, M, Co, epochs=14, dim=96, bs=64, lr=2e-3, mask_p=0.25, seed=0):
    torch.manual_seed(seed)
    model = Inspector(dim=dim).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    n = len(E)
    for ep in range(epochs):
        model.train()
        tot = cnt = 0.0
        for b in batches(n, bs, seed=seed * 100 + ep):
            e, m, c = t(E[b]), t(M[b]), t(Co[b])
            # denoising: hide a fraction of present edges from the ENCODER,
            # then ask the decoder to reproduce every present edge.
            drop = (torch.rand_like(m) > mask_p).float() * m
            _, pred = model(e * drop.unsqueeze(-1), drop, c * drop.unsqueeze(-1))
            loss = (((pred - e) ** 2).mean(-1) * m).sum() / m.sum().clamp(min=1)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += float(loss) * len(b); cnt += len(b)
        print(f"  inspector ep{ep:02d} loss {tot / cnt:.4f}", flush=True)
    return model


@torch.no_grad()
def inspector_forward(model, E, M, Co, bs=256):
    model.eval()
    Z, ERR = [], []
    for i in range(0, len(E), bs):
        e, m, c = t(E[i:i + bs]), t(M[i:i + bs]), t(Co[i:i + bs])
        z, pred = model(e, m, c)
        Z.append(z.numpy()); ERR.append(recon_error(pred, e, m).numpy())
    return np.concatenate(Z), np.concatenate(ERR)


def train_sentry(E, M, Zt, ERRt, epochs=16, dim=32, bs=64, lr=2e-3,
                 lam=4.0, seed=0):
    """Distil the ENCODER with an anomaly-weighted loss.

    L = E[(1 + lam * s_teacher) * ||project(f_student) - f_teacher||^2]

    Standard KD minimises mean divergence, which is precisely why rare signals
    wash out. Weighting by the teacher's own normalised anomaly score forces
    student capacity onto the windows the teacher found hardest.
    """
    torch.manual_seed(seed)
    model = Sentry(dim=dim, teacher_dim=Zt.shape[-1]).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    s = ERRt / (np.quantile(ERRt, 0.99) + 1e-9)
    s = np.clip(s, 0, 3.0).astype(np.float32)
    n = len(E)
    for ep in range(epochs):
        model.train(); tot = cnt = 0.0
        for b in batches(n, bs, seed=seed * 200 + ep):
            e, m = t(E[b]), t(M[b])
            zt, w = t(Zt[b]), t(s[b])
            _, proj = model(e, m)
            per = ((proj - zt) ** 2).mean(-1)             # (B,W)
            loss = ((1.0 + lam * w) * per).mean()
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += float(loss) * len(b); cnt += len(b)
        print(f"  sentry    ep{ep:02d} loss {tot / cnt:.4f}", flush=True)
    return model


@torch.no_grad()
def sentry_forward(model, E, M, bs=512):
    model.eval(); Z = []
    for i in range(0, len(E), bs):
        z, _ = model(t(E[i:i + bs]), t(M[i:i + bs]))
        Z.append(z.numpy())
    return np.concatenate(Z)


class Head(nn.Module):
    """Shared body for router A (regress teacher score) and C (deferral)."""
    def __init__(self, din, n_extra=0):
        super().__init__(); self.net = _mlp(din + n_extra, 64, 1)

    def forward(self, z, extra=None):
        x = z if extra is None else torch.cat([z, extra], -1)
        return self.net(x).squeeze(-1)


def train_head(Zs, target, extra=None, epochs=30, lr=3e-3, bs=256,
               loss="mse", seed=0):
    torch.manual_seed(seed)
    din, nx = Zs.shape[-1], (0 if extra is None else extra.shape[-1])
    head = Head(din, nx).to(DEV)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=1e-4)
    Zf = Zs.reshape(-1, din)
    yf = target.reshape(-1)
    Ef = None if extra is None else extra.reshape(-1, nx)
    fn = nn.MSELoss() if loss == "mse" else nn.BCEWithLogitsLoss()
    for ep in range(epochs):
        for b in batches(len(Zf), bs, seed=seed * 300 + ep):
            pred = head(t(Zf[b]), None if Ef is None else t(Ef[b]))
            l = fn(pred, t(yf[b]))
            opt.zero_grad(); l.backward(); opt.step()
    return head


@torch.no_grad()
def head_forward(head, Zs, extra=None):
    din = Zs.shape[-1]
    p = head(t(Zs.reshape(-1, din)),
             None if extra is None else t(extra.reshape(-1, extra.shape[-1])))
    return p.numpy().reshape(Zs.shape[:-1])


# ---------------------------------------------------------------- evaluation
def budget_curve(score, teacher_flag, attack, budgets):
    """At each escalation budget, what fraction of teacher-flagged (and of
    true-attack) windows do we recover? Threshold is the budget quantile."""
    out = []
    # Rank-based selection: robust to score ties, which a quantile threshold
    # is not (a mass of identical scores makes `score >= thr` select
    # everything and silently reports 100% recall).
    order = np.argsort(-np.asarray(score, dtype=np.float64), kind="stable")
    n = len(score)
    for b in budgets:
        k = max(1, int(round(b * n)))
        esc = np.zeros(n, dtype=bool)
        esc[order[:k]] = True
        rec_t = float(esc[teacher_flag].mean()) if teacher_flag.any() else float("nan")
        rec_a = float(esc[attack].mean()) if attack.any() else float("nan")
        out.append(dict(budget=float(b), actual=float(esc.mean()),
                        recall_teacher=rec_t, recall_attack=rec_a))
    return out

In [ ]:
%%writefile netsentinel_v2/cost_model.py
"""Inspector-hour cost model.

    GPU-host-days = H*D                (one-time commissioning)
                  + H*(r + e)*365      (ongoing: random sampling + escalation)
    vs inspect-all = H*365

Steady state should require ZERO GPU: the Sentry is a small encoder plus a
Mahalanobis/deferral evaluation, both CPU. Measured parameter counts and
per-window latencies are filled in by the experiment so the number quoted is
ours, not a citation.

Detection-latency bound (V2_HARDENING B6, corrected by G8):
    ETTE = 1 / (r * P(detect | inspected))     -- NOT 1/r.
If the attacker is quiet in most windows, an Inspector visit may see nothing.
"""

from __future__ import annotations


def gpu_host_days(H, D, r, e, days=365):
    one_time = H * D
    ongoing = H * (r + e) * days
    return dict(one_time=one_time, ongoing_per_year=ongoing,
                inspect_all_per_year=H * days,
                steady_state_reduction=1.0 - ongoing / (H * days))


def ette_days(r, p_detect_given_inspected, windows_per_day=24,
              p_visible_per_window=0.0):
    """Expected time to escalation, in days.

    Two paths: the Sentry sees a deviation (fast), or random sampling catches
    it (slow bound). Reported separately -- an honest bound beats a vague
    reassurance, and it is what a buyer puts in a risk register.
    """
    out = {"sampling_only_days": float("inf")}
    denom = r * max(p_detect_given_inspected, 1e-9)
    if denom > 0:
        out["sampling_only_days"] = 1.0 / denom
    if p_visible_per_window > 0:
        out["anomaly_path_days"] = 1.0 / (p_visible_per_window * windows_per_day)
    return out


def summarise(H=1000, D=14, r=1 / 30, e=0.02):
    c = gpu_host_days(H, D, r, e)
    lines = [
        f"hosts={H}  commissioning={D}d  sample_rate={r:.4f}/host-day  escalation={e:.4f}/host-day",
        f"  inspect-all              : {c['inspect_all_per_year']:>10,.0f} GPU-host-days/yr",
        f"  NetSentinel one-time     : {c['one_time']:>10,.0f} GPU-host-days",
        f"  NetSentinel ongoing      : {c['ongoing_per_year']:>10,.0f} GPU-host-days/yr",
        f"  steady-state reduction   : {c['steady_state_reduction'] * 100:>10.1f} %",
    ]
    return c, "\n".join(lines)

In [ ]:
%%writefile netsentinel_v2/zeek_loader.py
"""Zeek logs -> the same (host, day, window, category, feature) tensor the
generator produces. This is the seam that lets real traffic replace `synth`
without touching models.py, baseline.py or train.py.

Input: a directory of Zeek logs (TSV or JSON). Uses conn.log for volume and
timing, ssl.log for SNI and dns.log for query names so the Service Category
Resolver has a hostname to work with. Where both are missing (ECH, DoH, raw IP)
the flow lands in `Unknown_External` — which is deliberate: an unknown-category
transition is itself informative, and pretending otherwise hides how much
visibility you have actually lost.

Works on: Stratosphere Normal Captures (Bro/Zeek format), CTU-13, IoT-23, and
your own `zeek -i <iface>` capture. See DATASETS.md.
"""

from __future__ import annotations

import gzip, io, json, os, re
from collections import defaultdict

import numpy as np

from .categories import CAT_INDEX, N_CATEGORIES, N_EDGE_FEATURES
from .synth import _edge_row, WINDOWS_PER_DAY

# --- Service Category Resolver (deterministic, taxonomy-driven) ------------
# Extend from the LOTS Project taxonomy; keep it a lookup, never a model.
RULES = [
    ("Recon_API",       r"(ip-api\.com|ipify\.org|ifconfig\.me|icanhazip|ipinfo\.io|myexternalip)"),
    ("Code_Repo_Paste", r"(github|githubusercontent|gitlab|bitbucket|pastebin|paste\.ee|gist\.)"),
    ("Messaging_API",   r"(telegram|discord|slack\.com|teams\.microsoft|graph\.microsoft|mattermost)"),
    ("Cloud_Storage",   r"(drive\.google|docs\.google|dropbox|onedrive|1drv\.ms|s3[.-].*amazonaws|box\.com|mega\.nz|storage\.googleapis)"),
    ("CI_CD",           r"(circleci|travis-ci|jenkins|actions\.github|codecov|docker\.io|ghcr\.io|pypi\.org|registry\.npmjs)"),
    ("Sync",            r"(sync\.|clientsync|dropboxusercontent|apple-cloudkit|icloud)"),
]
_COMPILED = [(c, re.compile(p, re.I)) for c, p in RULES]
_PRIVATE = re.compile(r"^(10\.|192\.168\.|172\.(1[6-9]|2\d|3[01])\.|127\.|169\.254\.|fe80:|::1)")


def resolve_category(hostname: str | None, dest_ip: str | None) -> str:
    if hostname:
        for cat, rx in _COMPILED:
            if rx.search(hostname):
                return cat
        return "Browse"
    if dest_ip and _PRIVATE.match(dest_ip):
        return "Internal"
    return "Unknown_External"      # ECH / DoH / raw-IP: visibility genuinely lost


# --- Zeek log reading (TSV with #fields header, or JSON lines) -------------
def _open(path):
    return gzip.open(path, "rt", errors="replace") if path.endswith(".gz") \
        else open(path, "rt", errors="replace")


def read_zeek(path: str):
    """Yield dicts, one per record. Handles Zeek TSV and JSON-lines."""
    if not os.path.exists(path):
        return
    with _open(path) as f:
        fields, sep = None, "\t"
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            if line.startswith("#"):
                if line.startswith("#fields"):
                    fields = line.split("\t")[1:]
                elif line.startswith("#separator"):
                    s = line.split(" ", 1)[1].strip()
                    sep = s.encode().decode("unicode_escape") if s.startswith("\\") else s
                continue
            if fields is None:                       # JSON-lines
                try:
                    yield json.loads(line)
                except json.JSONDecodeError:
                    pass
                continue
            parts = line.split(sep)
            if len(parts) == len(fields):
                yield dict(zip(fields, parts))


def _num(v, default=0.0):
    try:
        if v in ("-", "(empty)", "", None):
            return default
        return float(v)
    except (TypeError, ValueError):
        return default


def load_dir(zeek_dir: str, window_seconds: int = 3600,
             min_windows_per_host: int = 48, max_hosts: int | None = None):
    """Build the tensors from one directory of Zeek logs.

    Returns the same dict shape as synth.generate():
      edges (H,D,W,C,F)  mask (H,D,W,C)  host_ids, and `hostnames`.
    `is_attack_day` is all-False — real benign capture carries no attack labels,
    which is the point: the Inspector is an anomaly model and never needs them.
    """
    conn = os.path.join(zeek_dir, "conn.log")
    for alt in ("conn.log.gz", "conn.log.labeled"):
        if not os.path.exists(conn) and os.path.exists(os.path.join(zeek_dir, alt)):
            conn = os.path.join(zeek_dir, alt)

    # uid -> hostname, from ssl (SNI) then dns (query)
    uid_host: dict[str, str] = {}
    for rec in read_zeek(os.path.join(zeek_dir, "ssl.log")):
        sn = rec.get("server_name") or rec.get("server_name.")
        if sn and sn != "-":
            uid_host[rec.get("uid", "")] = sn
    ip_host: dict[str, str] = {}
    for rec in read_zeek(os.path.join(zeek_dir, "dns.log")):
        q, ans = rec.get("query"), rec.get("answers")
        if q and q != "-" and ans and ans != "-":
            for a in str(ans).split(","):
                if re.match(r"^\d+\.\d+\.\d+\.\d+$", a.strip()):
                    ip_host[a.strip()] = q

    # (host, abs_window, category) -> list of flow tuples
    buckets: dict[tuple, list] = defaultdict(list)
    t_min = None
    for rec in read_zeek(conn):
        ts = _num(rec.get("ts"))
        if ts <= 0:
            continue
        src, dst = rec.get("id.orig_h"), rec.get("id.resp_h")
        if not src or not dst:
            continue
        t_min = ts if t_min is None else min(t_min, ts)
        name = uid_host.get(rec.get("uid", "")) or ip_host.get(dst)
        cat = resolve_category(name, dst)
        buckets[(src, int(ts // window_seconds), cat)].append((
            ts,
            _num(rec.get("orig_bytes")),
            _num(rec.get("resp_bytes")),
            _num(rec.get("duration")),
            dst,
        ))
    if not buckets:
        raise RuntimeError(f"no conn.log records found under {zeek_dir}")

    hosts = sorted({k[0] for k in buckets})
    counts = defaultdict(set)
    for (h, w, _c) in buckets:
        counts[h].add(w)
    hosts = [h for h in hosts if len(counts[h]) >= min_windows_per_host]
    if not hosts:                        # short captures: keep everything
        hosts = sorted({k[0] for k in buckets})
    if max_hosts:
        hosts = sorted(hosts, key=lambda h: -len(counts[h]))[:max_hosts]
    hidx = {h: i for i, h in enumerate(hosts)}

    w0 = min(k[1] for k in buckets)
    w1 = max(k[1] for k in buckets)
    n_windows = w1 - w0 + 1
    n_days = max(1, int(np.ceil(n_windows / WINDOWS_PER_DAY)))

    H, D, W, C, F = len(hosts), n_days, WINDOWS_PER_DAY, N_CATEGORIES, N_EDGE_FEATURES
    E = np.zeros((H, D, W, C, F), dtype=np.float32)
    M = np.zeros((H, D, W, C), dtype=np.float32)
    rng = np.random.default_rng(0)

    for (host, aw, cat), flows in buckets.items():
        if host not in hidx:
            continue
        off = aw - w0
        d, w = divmod(off, WINDOWS_PER_DAY)
        if d >= D:
            continue
        flows.sort()
        ts = np.array([f[0] for f in flows])
        iats = np.diff(ts) if ts.size > 1 else np.array([0.0])
        up = float(sum(f[1] for f in flows))
        down = float(sum(f[2] for f in flows))
        dur = float(np.mean([f[3] for f in flows]))
        distinct = len({f[4] for f in flows}) / max(len(flows), 1)
        E[hidx[host], d, w, CAT_INDEX[cat]] = _edge_row(
            rng, len(flows), up, down, iats, distinct, dur)
        M[hidx[host], d, w, CAT_INDEX[cat]] = 1.0

    return dict(edges=E, mask=M,
                is_attack_day=np.zeros((H, D), dtype=bool),
                stealth=np.zeros(H, dtype=np.float32),
                roles=np.array(["unknown"] * H),
                host_ids=np.arange(H), hostnames=np.array(hosts),
                source=zeek_dir)


def summarise(d):
    E, M = d["edges"], d["mask"]
    from .categories import CATEGORIES
    per_cat = M.sum(axis=(0, 1, 2))
    lines = [f"hosts={E.shape[0]}  days={E.shape[1]}  live windows="
             f"{int((M.sum(-1) > 0).sum())}  density={M.mean():.3f}"]
    for c, n in sorted(zip(CATEGORIES, per_cat), key=lambda x: -x[1]):
        if n:
            lines.append(f"    {c:18s} {int(n):>8,}")
    return "\n".join(lines)

In [ ]:
%%writefile netsentinel_v2/__init__.py


In [ ]:
%%writefile run_experiment.py
#!/usr/bin/env python3
"""Run the full V2 experiment and emit results.json.

  python run_experiment.py --seeds 3 --hosts 300 --days 24
"""
from __future__ import annotations

import argparse, json, time, sys
import numpy as np
import torch

from netsentinel_v2 import synth, train as T, cost_model
from netsentinel_v2.baseline import HostBaseline
from netsentinel_v2.diagnostics import geometry_report, verdict

BUDGETS = [0.005, 0.01, 0.02, 0.03, 0.05, 0.075, 0.10, 0.15, 0.20, 0.30]


def count_params(m):
    return sum(p.numel() for p in m.parameters())


def roc_auc(score, label):
    from sklearn.metrics import roc_auc_score
    label = np.asarray(label).astype(bool)
    if label.all() or not label.any():
        return float("nan")
    return float(roc_auc_score(label, np.asarray(score, dtype=np.float64)))


def run_seed(seed, hosts, days, epochs_t, epochs_s, stealth_range, verbose=True):
    t0 = time.time()
    d = synth.generate(n_hosts=hosts, n_days=days, seed=seed,
                       stealth_range=stealth_range)
    E, M, A = d["edges"], d["mask"], d["is_attack_day"]
    H, D, W, C, F = E.shape

    Co = T.compute_cohort(E, M)

    # ---- strictly monotonic temporal split (no random splits) -------------
    c_end, v_end = D // 2, int(D * 0.66)
    sl = lambda a, b: (slice(None), slice(a, b))
    flat = lambda X, a, b: X[:, a:b].reshape(-1, *X.shape[2:])
    hidx = lambda a, b: np.repeat(np.arange(H), b - a)

    Ec, Mc, Cc = flat(E, 0, c_end), flat(M, 0, c_end), flat(Co, 0, c_end)
    Ev, Mv, Cv = flat(E, c_end, v_end), flat(M, c_end, v_end), flat(Co, c_end, v_end)
    Et, Mt, Ct = flat(E, v_end, D), flat(M, v_end, D), flat(Co, v_end, D)
    hv, ht = hidx(c_end, v_end), hidx(v_end, D)
    hc = hidx(0, c_end)

    Ec, mu, sd = T.standardise(Ec, Mc)
    Ev, *_ = T.standardise(Ev, Mv, mu, sd)
    Et, *_ = T.standardise(Et, Mt, mu, sd)

    if verbose:
        print(f"[seed {seed}] commissioning {Ec.shape[0]} host-days | "
              f"val {Ev.shape[0]} | test {Et.shape[0]}", flush=True)

    # ---- Inspector --------------------------------------------------------
    insp = T.train_inspector(Ec, Mc, Cc, epochs=epochs_t, seed=seed)
    Zc_t, Ec_err = T.inspector_forward(insp, Ec, Mc, Cc)
    Zv_t, Ev_err = T.inspector_forward(insp, Ev, Mv, Cv)
    Zt_t, Et_err = T.inspector_forward(insp, Et, Mt, Ct)

    # Teacher's decision, calibrated on COMMISSIONING only.
    thr_t = float(np.quantile(Ec_err, 0.99))
    flag_v, flag_t = Ev_err >= thr_t, Et_err >= thr_t

    # ---- Sentry (encoder distillation only) -------------------------------
    sen = T.train_sentry(Ec, Mc, Zc_t, Ec_err, epochs=epochs_s, seed=seed)
    Zc_s = T.sentry_forward(sen, Ec, Mc)
    Zv_s = T.sentry_forward(sen, Ev, Mv)
    Zt_s = T.sentry_forward(sen, Et, Mt)

    # Windows with no traffic at all carry no decision and must not be scored;
    # including them inflates every recall number at high budget.
    live_t = (Mt.sum(-1) > 0)
    live_v = (Mv.sum(-1) > 0)

    # ---- per-host baselines in student space (G3/G4, trimmed) -------------
    # Fit on LIVE commissioning windows only. An empty window carries no
    # decision; including them puts a zero-variance spike in the baseline and
    # collapses the distance distribution into a mass of ties.
    live_c = (Mc.sum(-1) > 0)
    bl = {}
    for h in range(H):
        rows = np.where(hc == h)[0]
        Z = Zc_s[rows].reshape(-1, Zc_s.shape[-1])
        lv = live_c[rows].reshape(-1)
        err = Ec_err[rows].reshape(-1)
        if lv.sum() >= 20:
            Z, err = Z[lv], err[lv]
        bl[h] = HostBaseline(n_components=3, trim_frac=0.05).fit(Z, err)

    def maha(Z, hidx_):
        out = np.zeros(Z.shape[:2], dtype=np.float32)
        for h in np.unique(hidx_):
            r = np.where(hidx_ == h)[0]
            out[r] = bl[h].score(Z[r].reshape(-1, Z.shape[-1])).reshape(len(r), -1)
        return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

    Mv_d, Mt_d = maha(Zv_s, hv), maha(Zt_s, ht)
    Mc_d = maha(Zc_s, hc)

    # ---- Router A: distilled DETECTOR (the original design) ---------------
    headA = T.train_head(Zc_s, (Ec_err / (thr_t + 1e-9)).astype(np.float32),
                         loss="mse", seed=seed)
    sA_t = T.head_forward(headA, Zt_s)

    # ---- Router B: encoder + Mahalanobis (V2_HARDENING B1) ----------------
    sB_t = Mt_d

    # ---- Router C: deferral head (V2_HARDENING G5) ------------------------
    # label is free: during commissioning we ran BOTH models.
    yC = (Ec_err >= thr_t).astype(np.float32)
    extra_c = np.stack([Mc_d, np.log1p(Mc_d), Mc.sum(-1)], -1).astype(np.float32)
    extra_t = np.stack([Mt_d, np.log1p(Mt_d), Mt.sum(-1)], -1).astype(np.float32)
    headC = T.train_head(Zc_s, yC, extra=extra_c, loss="bce", seed=seed)
    sC_t = T.head_forward(headC, Zt_s, extra_t)

    # ---- attack labels (indicative only -- synthetic) ---------------------
    atk_day = A[:, v_end:D].reshape(-1)
    atk_win = np.repeat(atk_day[:, None], W, axis=1) & live_t

    # ---- G1 diagnostic on LIVE windows only -------------------------------
    lf = live_t.reshape(-1)
    geo = geometry_report(Zt_t.reshape(-1, Zt_t.shape[-1])[lf],
                          Zt_s.reshape(-1, Zt_s.shape[-1])[lf],
                          Et_err.reshape(-1)[lf],
                          sA_t.reshape(-1)[lf])
    v_code, v_msg = verdict(geo)

    # ---- Is the ORACLE any good? If the Inspector cannot separate attacks,
    # every agreement number below is agreement with noise. Report it first.
    teach_auc = roc_auc(Et_err.reshape(-1)[lf], atk_win.reshape(-1)[lf])

    sel = lambda s: s.reshape(-1)[lf]
    curves = {
        "A_distilled_detector":  T.budget_curve(sel(sA_t), sel(flag_t), sel(atk_win), BUDGETS),
        "B_encoder_mahalanobis": T.budget_curve(sel(sB_t), sel(flag_t), sel(atk_win), BUDGETS),
        "C_deferral_head":       T.budget_curve(sel(sC_t), sel(flag_t), sel(atk_win), BUDGETS),
    }
    router_auc = {
        "A_distilled_detector":  roc_auc(sel(sA_t), sel(flag_t)),
        "B_encoder_mahalanobis": roc_auc(sel(sB_t), sel(flag_t)),
        "C_deferral_head":       roc_auc(sel(sC_t), sel(flag_t)),
    }

    res = dict(
        seed=seed, geometry=geo, verdict=[v_code, v_msg], curves=curves,
        router_auc_vs_teacher=router_auc,
        teacher_auc_vs_attack=teach_auc,
        params=dict(inspector=count_params(insp), sentry=count_params(sen),
                    deferral_head=count_params(headC),
                    compression=count_params(insp) / max(count_params(sen), 1)),
        teacher_flag_rate_test=float(sel(flag_t).mean()),
        attack_window_rate_test=float(sel(atk_win).mean()),
        live_window_frac=float(lf.mean()),
        runtime_s=time.time() - t0,
    )
    return res


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--seeds", type=int, default=3)
    ap.add_argument("--hosts", type=int, default=300)
    ap.add_argument("--days", type=int, default=24)
    ap.add_argument("--epochs-teacher", type=int, default=14)
    ap.add_argument("--epochs-student", type=int, default=16)
    ap.add_argument("--stealth-max", type=float, default=1.0)
    ap.add_argument("--out", default="results.json")
    a = ap.parse_args()

    runs = [run_seed(s, a.hosts, a.days, a.epochs_teacher, a.epochs_student,
                     (0.0, a.stealth_max)) for s in range(a.seeds)]

    # aggregate (multi-seed mean +/- std, per the benchmark protocol)
    agg = {}
    for router in runs[0]["curves"]:
        agg[router] = []
        for i, b in enumerate(BUDGETS):
            rt = [r["curves"][router][i]["recall_teacher"] for r in runs]
            ra = [r["curves"][router][i]["recall_attack"] for r in runs]
            agg[router].append(dict(
                budget=b,
                recall_teacher_mean=float(np.mean(rt)), recall_teacher_std=float(np.std(rt)),
                recall_attack_mean=float(np.nanmean(ra)), recall_attack_std=float(np.nanstd(ra))))

    cost, cost_txt = cost_model.summarise()
    geo_mean = {k: float(np.mean([r["geometry"][k] for r in runs]))
                for k in runs[0]["geometry"]}
    # verdict from the MEAN across seeds, not seed 0
    v_code, v_msg = verdict(geo_mean)

    # lift over random routing -- the number that says whether routing works
    best = max(agg, key=lambda r: agg[r][BUDGETS.index(0.05)]["recall_teacher_mean"])
    lift = {f"{b:.3f}": {r: (agg[r][i]["recall_teacher_mean"] / b) for r in agg}
            for i, b in enumerate(BUDGETS)}

    out = dict(runs=runs, aggregate=agg, budgets=BUDGETS,
               geometry_mean=geo_mean, verdict=[v_code, v_msg],
               lift_over_random=lift, best_router_at_5pct=best,
               oracle_auc=float(np.nanmean([r["teacher_auc_vs_attack"] for r in runs])),
               router_auc={k: float(np.nanmean([r["router_auc_vs_teacher"][k] for r in runs]))
                           for k in runs[0]["router_auc_vs_teacher"]},
               cost=cost, cost_text=cost_txt, params=runs[0]["params"])
    with open(a.out, "w") as f:
        json.dump(out, f, indent=2)

    # ---- console report ---------------------------------------------------
    print("\n" + "=" * 72)
    print("ORACLE QUALITY — can the Inspector separate attacks at all?")
    ta = [r["teacher_auc_vs_attack"] for r in runs]
    print(f"  Inspector AUC vs (synthetic) attack windows: "
          f"{np.nanmean(ta):.3f} ± {np.nanstd(ta):.3f}")
    if np.nanmean(ta) < 0.65:
        print("  !! WEAK ORACLE. Teacher-agreement numbers below are agreement")
        print("     with a detector that barely detects. Do not quote them.")
    print("\nROUTER AUC vs TEACHER DECISION")
    for r_ in runs[0]["router_auc_vs_teacher"]:
        vals = [r["router_auc_vs_teacher"][r_] for r in runs]
        print(f"  {r_:24s} {np.nanmean(vals):.3f} ± {np.nanstd(vals):.3f}")
    print("\nG1 GEOMETRY DIAGNOSTIC (go/no-go for the whole B1 fix)")
    for k, v in out["geometry_mean"].items():
        print(f"  {k:24s} {v:.4f}")
    print(f"  VERDICT: {out['verdict'][0]} — {out['verdict'][1]}")
    print("\nMODEL SIZE")
    p = out["params"]
    print(f"  Inspector {p['inspector']:,} params | Sentry {p['sentry']:,} | "
          f"compression {p['compression']:.1f}x")
    print("\nRECALL OF TEACHER-FLAGGED WINDOWS @ ESCALATION BUDGET (mean±std, n=%d seeds)"
          % a.seeds)
    hdr = "  budget | " + " | ".join(f"{r[:22]:>22s}" for r in agg)
    print(hdr); print("  " + "-" * (len(hdr) - 2))
    for i, b in enumerate(BUDGETS):
        row = f"  {b * 100:5.1f}% | "
        row += " | ".join(f"{agg[r][i]['recall_teacher_mean'] * 100:16.1f} ±{agg[r][i]['recall_teacher_std'] * 100:4.1f}"
                          for r in agg)
        print(row)
    print("\nRECALL OF (SYNTHETIC) ATTACK WINDOWS — INDICATIVE ONLY")
    for i, b in enumerate(BUDGETS):
        if b not in (0.01, 0.05, 0.10, 0.20):
            continue
        row = f"  {b * 100:5.1f}% | "
        row += " | ".join(f"{agg[r][i]['recall_attack_mean'] * 100:16.1f} ±{agg[r][i]['recall_attack_std'] * 100:4.1f}"
                          for r in agg)
        print(row)
    print("\nLIFT OVER RANDOM ROUTING (recall / budget) — best router: %s" % best)
    for i, b in enumerate(BUDGETS):
        if b in (0.01, 0.05, 0.10, 0.20):
            print(f"  {b*100:5.1f}% budget | " + " | ".join(
                f"{r.split('_')[0]}:{lift[f'{b:.3f}'][r]:5.1f}x" for r in agg))
    tgt = agg[best][BUDGETS.index(0.05)]["recall_teacher_mean"]
    print(f"\n  DESIGN TARGET CHECK: V2_HARDENING proposed '>=95% recall at <=5% budget'.")
    print(f"  Measured best at 5% budget: {tgt*100:.1f}%  ->  "
          f"{'MET' if tgt >= 0.95 else 'NOT MET'}. Do not quote the target as achieved.")
    print("\nCOST MODEL"); print(cost_txt)
    print("=" * 72)
    print(f"\nwrote {a.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile shift_test.py
#!/usr/bin/env python3
"""Distribution-shift stress test: how brittle is the learned "normal"?

The question this answers is the one a judge will ask:

    "Your Inspector learned normal from your own generator. What happens when
     you point it at a network it has never seen?"

Setup — the honest analogue of a real deployment:

    WORLD A  dev-heavy software org      -> the Inspector and Sentry are TRAINED here
    WORLD B  OT/plant-heavy org          -> same mechanism, different organisation:
                                            different role mix, slower human rhythms,
                                            lower data volumes, longer working day

The encoder is trained ONCE on world A and never sees world B traffic during
training. In world B it is deployed cold: baselines are re-fit on B's own
commissioning window (which is what actually happens at a new customer), and
thresholds calibrated on A are applied unchanged, because that is the mistake
a real deployment makes.

Reported degradations:
  * Inspector AUC vs attack           A -> B     (does the oracle survive?)
  * Router AUC vs Inspector           A -> B     (does routing survive?)
  * Recall @5% escalation budget      A -> B
  * kNN geometry overlap              A -> B
  * Threshold transfer: the flag-rate world A's 99th-percentile threshold
    actually produces in world B. If that number is not ~1%, calibration does
    NOT transfer and every deployment needs its own calibration window.
"""
from __future__ import annotations

import argparse, json, time
import numpy as np

from netsentinel_v2 import synth, train as T
from netsentinel_v2.baseline import HostBaseline
from netsentinel_v2.diagnostics import geometry_report
from run_experiment import roc_auc, count_params, BUDGETS


def prep(d):
    E, M = d["edges"], d["mask"]
    Co = T.compute_cohort(E, M)
    return E, M, Co, d["is_attack_day"]


def split(X, a, b):
    return X[:, a:b].reshape(-1, *X.shape[2:])


def run(seed, hosts, days, ep_t, ep_s):
    t0 = time.time()
    D = days
    c_end, v_end = D // 2, int(D * 0.66)

    A = synth.generate(n_hosts=hosts, n_days=days, seed=seed, world_cfg="A")
    B = synth.generate(n_hosts=hosts, n_days=days, seed=seed + 500, world_cfg="B")

    Ea, Ma, Coa, Aa = prep(A)
    Eb, Mb, Cob, Ab = prep(B)
    H = hosts

    # ---- world A: the training world ------------------------------------
    Ac, Mc, Cc = split(Ea, 0, c_end), split(Ma, 0, c_end), split(Coa, 0, c_end)
    At, Mt, Ct = split(Ea, v_end, D), split(Ma, v_end, D), split(Coa, v_end, D)
    Ac, mu, sd = T.standardise(Ac, Mc)
    At, *_ = T.standardise(At, Mt, mu, sd)

    # ---- world B: deployed cold. NOTE the standardiser is A's -----------
    # Re-deriving it on B would quietly hide part of the shift; a real
    # deployment ships the fitted preprocessing with the model.
    Bc, Nc, Dc = split(Eb, 0, c_end), split(Mb, 0, c_end), split(Cob, 0, c_end)
    Bt, Nt, Dt = split(Eb, v_end, D), split(Mb, v_end, D), split(Cob, v_end, D)
    Bc, *_ = T.standardise(Bc, Nc, mu, sd)
    Bt, *_ = T.standardise(Bt, Nt, mu, sd)

    hc = np.repeat(np.arange(H), c_end)
    ht = np.repeat(np.arange(H), D - v_end)

    print(f"[seed {seed}] training on world A ({Ac.shape[0]} host-days)", flush=True)
    insp = T.train_inspector(Ac, Mc, Cc, epochs=ep_t, seed=seed)
    Zc_t, Ec_err = T.inspector_forward(insp, Ac, Mc, Cc)
    Zat, Eat_err = T.inspector_forward(insp, At, Mt, Ct)
    Zbc, Ebc_err = T.inspector_forward(insp, Bc, Nc, Dc)
    Zbt, Ebt_err = T.inspector_forward(insp, Bt, Nt, Dt)

    thr = float(np.quantile(Ec_err, 0.99))          # calibrated on A only
    thr_b = float(np.quantile(Ebc_err, 0.99))       # what B's own calibration would be

    sen = T.train_sentry(Ac, Mc, Zc_t, Ec_err, epochs=ep_s, seed=seed)
    Zc_s = T.sentry_forward(sen, Ac, Mc)
    Zat_s = T.sentry_forward(sen, At, Mt)
    Zbc_s = T.sentry_forward(sen, Bc, Nc)
    Zbt_s = T.sentry_forward(sen, Bt, Nt)

    def baselines(Zs, mask_c, err, hidx):
        live = (mask_c.sum(-1) > 0)
        bl = {}
        for h in range(H):
            r = np.where(hidx == h)[0]
            Z = Zs[r].reshape(-1, Zs.shape[-1]); lv = live[r].reshape(-1)
            e = err[r].reshape(-1)
            if lv.sum() >= 20:
                Z, e = Z[lv], e[lv]
            bl[h] = HostBaseline(n_components=3, trim_frac=0.05).fit(Z, e)
        return bl

    def maha(bl, Z, hidx):
        out = np.zeros(Z.shape[:2], dtype=np.float32)
        for h in np.unique(hidx):
            r = np.where(hidx == h)[0]
            out[r] = bl[h].score(Z[r].reshape(-1, Z.shape[-1])).reshape(len(r), -1)
        return np.nan_to_num(out, nan=0., posinf=0., neginf=0.)

    bl_a = baselines(Zc_s, Mc, Ec_err, hc)
    bl_b = baselines(Zbc_s, Nc, Ebc_err, hc)   # re-fit on B — the deployment step

    res = {}
    for tag, (Zt_t, err, Zt_s, bl, mask, atk_day) in {
        "A": (Zat, Eat_err, Zat_s, bl_a, Mt, Aa),
        "B": (Zbt, Ebt_err, Zbt_s, bl_b, Nt, Ab),
    }.items():
        live = (mask.sum(-1) > 0); lf = live.reshape(-1)
        atk = (np.repeat(atk_day[:, v_end:D].reshape(-1)[:, None], mask.shape[1], 1) & live)
        flag = err >= thr                       # A's threshold, applied to both
        sB = maha(bl, Zt_s, ht)
        geo = geometry_report(Zt_t.reshape(-1, Zt_t.shape[-1])[lf],
                              Zt_s.reshape(-1, Zt_s.shape[-1])[lf],
                              err.reshape(-1)[lf], sB.reshape(-1)[lf])
        curve = T.budget_curve(sB.reshape(-1)[lf], flag.reshape(-1)[lf],
                               atk.reshape(-1)[lf], BUDGETS)
        res[tag] = dict(
            oracle_auc=roc_auc(err.reshape(-1)[lf], atk.reshape(-1)[lf]),
            router_auc=roc_auc(sB.reshape(-1)[lf], flag.reshape(-1)[lf]),
            recall_at_5=curve[BUDGETS.index(0.05)]["recall_teacher"],
            recall_attack_at_5=curve[BUDGETS.index(0.05)]["recall_attack"],
            knn=geo["knn_overlap@20"],
            flag_rate=float(flag.reshape(-1)[lf].mean()),
            attack_rate=float(atk.reshape(-1)[lf].mean()),
            curve=curve,
        )
    res["threshold_transfer"] = dict(
        thr_from_A=thr, thr_native_B=thr_b,
        ratio=thr_b / thr if thr else float("nan"),
        flag_rate_target=0.01,
        flag_rate_B_using_A_threshold=res["B"]["flag_rate"],
    )
    res["seed"] = seed
    res["runtime_s"] = time.time() - t0
    return res


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--seeds", type=int, default=3)
    ap.add_argument("--hosts", type=int, default=300)
    ap.add_argument("--days", type=int, default=24)
    ap.add_argument("--epochs-teacher", type=int, default=12)
    ap.add_argument("--epochs-student", type=int, default=14)
    ap.add_argument("--out", default="shift_results.json")
    a = ap.parse_args()

    runs = [run(s, a.hosts, a.days, a.epochs_teacher, a.epochs_student)
            for s in range(a.seeds)]
    agg = {}
    for w in ("A", "B"):
        agg[w] = {k: (float(np.nanmean([r[w][k] for r in runs])),
                      float(np.nanstd([r[w][k] for r in runs])))
                  for k in ("oracle_auc", "router_auc", "recall_at_5",
                            "recall_attack_at_5", "knn", "flag_rate")}
    tt = {k: float(np.nanmean([r["threshold_transfer"][k] for r in runs]))
          for k in ("ratio", "flag_rate_B_using_A_threshold")}
    out = dict(runs=runs, aggregate=agg, threshold_transfer=tt, budgets=BUDGETS,
               worlds={k: synth.WORLDS[k] for k in ("A", "B")})
    json.dump(out, open(a.out, "w"), indent=2)

    print("\n" + "=" * 74)
    print("DISTRIBUTION-SHIFT STRESS TEST — trained on world A, deployed cold to B")
    print("=" * 74)
    rows = [("Inspector AUC vs attack", "oracle_auc", 1),
            ("Router AUC vs Inspector", "router_auc", 1),
            ("Recall @5% budget", "recall_at_5", 100),
            ("Attack recall @5% (indicative)", "recall_attack_at_5", 100),
            ("kNN geometry overlap", "knn", 1),
            ("Flag rate (target 1.0%)", "flag_rate", 100)]
    print(f"  {'metric':34s} {'world A':>16s} {'world B':>16s}   change")
    print("  " + "-" * 72)
    for label, k, scale in rows:
        a_m, a_s = agg["A"][k]; b_m, b_s = agg["B"][k]
        d = (b_m - a_m) / a_m * 100 if a_m else float("nan")
        u = "%" if scale == 100 else ""
        print(f"  {label:34s} {a_m*scale:9.2f}{u} ±{a_s*scale:4.2f} "
              f"{b_m*scale:9.2f}{u} ±{b_s*scale:4.2f}   {d:+6.1f}%")
    print(f"\n  Threshold transfer: world B's native 99th-pct threshold is "
          f"{tt['ratio']:.2f}× world A's.")
    print(f"  Applying A's threshold in B flags "
          f"{tt['flag_rate_B_using_A_threshold']*100:.2f}% of windows "
          f"(target 1.00%).")
    print("=" * 74)
    print(f"\nwrote {a.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile make_charts.py
#!/usr/bin/env python3
"""Render the two Grand Finale charts from results.json.

Chart 1 (the money chart): recall of Inspector-flagged windows vs escalation
budget, three routers. This is the claim the panel can check.
Chart 2: the G1 geometry diagnostic, because the go/no-go belongs on a slide
next to the result it qualifies.

Palette: validated categorical slots 1-3 (blue/orange/aqua). Aqua sits below
3:1 on the light surface, so every series carries a direct label (the relief
rule) and identity never rests on colour alone.
"""
from __future__ import annotations

import json, sys
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SERIES = {                       # fixed slot order, never cycled
    "A_distilled_detector":  ("#2a78d6", "A · distilled detector\n(original V2 design)"),
    "B_encoder_mahalanobis": ("#eb6834", "B · encoder + Mahalanobis\n(V2_HARDENING B1)"),
    "C_deferral_head":       ("#1baf7a", "C · deferral head\n(V2_HARDENING G5)"),
}
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#fcfcfb"


def style(ax):
    ax.set_facecolor(SURFACE)
    ax.grid(True, color=GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color("#c3c2b7"); ax.spines[s].set_linewidth(1)
    ax.tick_params(colors=MUTED, labelsize=9, length=0)


def chart_budget(res, path):
    agg, budgets = res["aggregate"], np.array(res["budgets"]) * 100
    fig, ax = plt.subplots(figsize=(8.4, 5.0), dpi=200)
    fig.patch.set_facecolor(SURFACE); style(ax)

    ends = {}
    for key, (color, label) in SERIES.items():
        m = np.array([r["recall_teacher_mean"] for r in agg[key]]) * 100
        s = np.array([r["recall_teacher_std"] for r in agg[key]]) * 100
        ax.fill_between(budgets, m - s, m + s, color=color, alpha=0.13, linewidth=0)
        ax.plot(budgets, m, color=color, linewidth=2, zorder=3)
        ax.plot(budgets, m, "o", color=color, markersize=4.5,
                markeredgecolor=SURFACE, markeredgewidth=1.6, zorder=4)
        ends[key] = (m[-1], color, label)

    # de-collide the direct labels: each two-line label needs ~11 y-units
    order = sorted(ends, key=lambda k: -ends[k][0])
    placed: list[float] = []
    for key in order:
        y, color, label = ends[key]
        ly = y
        while any(abs(ly - p) < 11 for p in placed):
            ly -= 1.5
        placed.append(ly)
        ax.annotate(label, (budgets[-1], y), xytext=(9, (ly - y) * 4.6),
                    textcoords="offset points", color=color, fontsize=8.5,
                    va="center", linespacing=1.35, fontweight="medium")

    ax.plot(budgets, budgets, linestyle=(0, (4, 4)), color=MUTED,
            linewidth=1.2, zorder=2)
    ax.annotate("random routing", (budgets[-1], budgets[-1]), xytext=(8, -2),
                textcoords="offset points", color=MUTED, fontsize=8.5, va="center")

    ax.set_xlabel("Escalation budget — % of host-windows sent to the Inspector",
                  color="#52514e", fontsize=10, labelpad=9)
    ax.set_ylabel("Recall of Inspector-flagged windows (%)",
                  color="#52514e", fontsize=10, labelpad=9)
    ax.set_title("Can a cheap Sentry route well enough to keep the Inspector's ceiling?",
                 color=INK, fontsize=12.5, fontweight="semibold", loc="left", pad=14)
    ax.set_xlim(0, budgets[-1] * 1.02); ax.set_ylim(0, 100)
    ax.set_xticks([0, 5, 10, 15, 20, 25, 30])
    fig.text(0.008, 0.015,
             "Mean ±1σ over %d seeds · temporal train→val→test split · "
             "thresholds calibrated on validation only" % len(res["runs"]),
             color=MUTED, fontsize=7.6)
    fig.subplots_adjust(right=0.735, left=0.085, top=0.885, bottom=0.145)
    fig.savefig(path, facecolor=SURFACE)
    print("wrote", path)


def chart_geometry(res, path):
    g = res["geometry_mean"]
    keys = [k for k in g if k != "n"]
    vals = [g[k] for k in keys]
    names = {"knn_overlap@20": "kNN neighbourhood overlap\n(the real test)",
             "trustworthiness": "Trustworthiness",
             "spearman_score": "Score correlation\n(the weak test)"}
    fig, ax = plt.subplots(figsize=(7.6, 3.3), dpi=200)
    fig.patch.set_facecolor(SURFACE); style(ax)
    ypos = np.arange(len(keys))[::-1]
    colors = ["#d03b3b" if (k == "knn_overlap@20" and v < 0.30) else "#2a78d6"
              for k, v in zip(keys, vals)]
    ax.barh(ypos, [abs(v) for v in vals], height=0.42, color=colors, zorder=3)
    for y, v in zip(ypos, vals):
        ax.text(abs(v) + 0.015, y, f"{v:.2f}", va="center", fontsize=10,
                color=INK, fontweight="semibold")
    ax.axvline(0.30, color=MUTED, linestyle=(0, (4, 4)), linewidth=1.2, zorder=2)
    ax.text(0.305, ypos[0] + 0.42, "0.30 = go/no-go", color=MUTED, fontsize=8)
    ax.set_yticks(ypos); ax.set_yticklabels([names[k] for k in keys], fontsize=9,
                                            color="#52514e", linespacing=1.3)
    ax.set_xlim(0, max(1.0, max(abs(v) for v in vals) * 1.25))
    ax.set_title("G1 diagnostic — does the Sentry preserve the Inspector's local geometry?",
                 color=INK, fontsize=11.5, fontweight="semibold", loc="left", pad=12)
    fig.text(0.008, 0.02, "VERDICT: %s — %s" % (res["verdict"][0], res["verdict"][1][:110]),
             color=MUTED, fontsize=7.6, wrap=True)
    fig.subplots_adjust(left=0.285, right=0.97, top=0.83, bottom=0.16)
    fig.savefig(path, facecolor=SURFACE)
    print("wrote", path)


if __name__ == "__main__":
    res = json.load(open(sys.argv[1] if len(sys.argv) > 1 else "results.json"))
    chart_budget(res, "chart_escalation_budget.png")
    chart_geometry(res, "chart_geometry_diagnostic.png")

## 2 · Smoke test (~30 s)
Small and fast. Confirms the whole path runs before you spend real time.

In [ ]:
!python run_experiment.py --seeds 1 --hosts 60 --days 16 --epochs-teacher 3 --epochs-student 3 --out smoke.json

## 3 · The main experiment

3 seeds, 300 hosts, 24 days. ~2 min/seed on CPU.

Protocol enforced in code: strictly monotonic **train → validate → test in time** (no random
splits), thresholds calibrated on **validation only**, multi-seed ± σ, rank-based budget
selection so score ties can't fake 100% recall.

In [ ]:
!python run_experiment.py --seeds 3 --hosts 300 --days 24 --epochs-teacher 12 --epochs-student 14 --out results.json

### The chart

In [ ]:
!pip -q install matplotlib >/dev/null 2>&1
!python make_charts.py results.json
from IPython.display import Image, display
display(Image('chart_escalation_budget.png'))
display(Image('chart_geometry_diagnostic.png'))

## 4 · Distribution-shift stress test

The question a judge will ask: *your model learned normal from your own generator — what
happens on a network it has never seen?*

**World A** (dev-heavy software org) trains the encoder. **World B** (OT/plant-heavy: different
role mix, slower human rhythms, lower volumes, longer working day) is deployed to cold —
baselines re-fit on B's own commissioning window, but **A's threshold and A's standardiser
applied unchanged**, because that is the mistake a real deployment makes.

Watch the **flag rate** row. If A's 99th-percentile threshold doesn't produce ~1% in B,
calibration does not transfer and every deployment needs its own calibration window.

In [ ]:
!python shift_test.py --seeds 3 --hosts 300 --days 24 --epochs-teacher 12 --epochs-student 14 --out shift_results.json

## 5 · Real traffic (optional — needs Internet ON)

Everything above is synthetic. This section swaps in **real benign capture** so the encoder
learns what traffic actually looks like.

Sources and their limits are in `DATASETS.md`. Short version: Stratosphere Normal Captures are
real human/host activity in Zeek format, free — good for **encoder pretraining**, useless for
multi-day per-host baselines (the captures are only hours long). For baselines you need your
own machines recording for weeks.

In [ ]:
# Stratosphere Normal Captures — real benign, Bro/Zeek format.
# Browse https://www.stratosphereips.org/datasets-normal for the full list.
BASE = 'https://mcfp.felk.cvut.cz/publicDatasets'
CAPTURES = ['CTU-Normal-7', 'CTU-Normal-12']   # add more once one works

import os, subprocess
os.makedirs('data', exist_ok=True)
for c in CAPTURES:
    print('---', c)
    # -A '*.log*' grabs the Zeek/Bro logs and skips the large pcaps
    subprocess.run(['wget','-q','-r','-np','-nH','--cut-dirs=1','-R','index.html*',
                    '-A','*.log,*.log.gz,*.labeled',
                    f'{BASE}/{c}/','-P','data'], check=False)
!find data -name '*.log*' | head -30

In [ ]:
# Point this at whichever directory actually contains conn.log
import glob, os
cands = [os.path.dirname(p) for p in glob.glob('data/**/conn.log*', recursive=True)]
print('found conn.log in:', cands)

from netsentinel_v2 import zeek_loader as zl
if cands:
    real = zl.load_dir(cands[0], min_windows_per_host=1)
    print(zl.summarise(real))
else:
    print('No conn.log found. Either Internet is off, or this capture ships pcap only —',
          'run `zeek -r <file>.pcap` on it first, or pick another capture.')

### Train the encoder on the real capture

Same models, different data source — that's the whole point of the loader. Note there are no
attack labels here and none are needed: the Inspector is an anomaly model that learns normal.
What you get out is a **representation**; the per-host baselines still come from commissioning
at deployment time.

In [ ]:
import numpy as np, torch
from netsentinel_v2 import train as T

if cands:
    E, M = real['edges'], real['mask']
    Co = T.compute_cohort(E, M)
    flat = lambda X: X.reshape(-1, *X.shape[2:])
    Ef, Mf, Cf = flat(E), flat(M), flat(Co)
    Ef, mu, sd = T.standardise(Ef, Mf)
    print('host-days:', Ef.shape[0])
    insp = T.train_inspector(Ef, Mf, Cf, epochs=8, seed=0)
    Z, err = T.inspector_forward(insp, Ef, Mf, Cf)
    print('embedding', Z.shape, '| recon error mean %.4f p99 %.4f'
          % (err.mean(), np.quantile(err, 0.99)))
    torch.save(insp.state_dict(), 'inspector_real_pretrained.pt')
    print('saved inspector_real_pretrained.pt — this is the transferable part')

## 6 · What to take away

- `results.json` — router comparison, the number you quote
- `shift_results.json` — how much of it survives a new network
- `chart_escalation_budget.png` — the slide
- `inspector_real_pretrained.pt` — encoder weights from real traffic

**Do not quote** ≥95% recall (measured 37.5% at a 5% budget), and mark every attack-recall
number indicative-only while the generator is still the attack source.